# t04_1：人工排除 ticker 後的診斷統計

## 定義與假設
- 只依已審核 CSV 的 `exclude=1` 排除；未填 reason 仍接受並統計，不新增自動高頻詞或人工猜測規則。
- 使用審核 run_summary 指定的 03 threads.parquet，以確保前後比較與審核資料一致。
- **mention 定義：**一個 thread 中一支 unique ticker 計一次；不是原始留言詞次或 comment–ticker 次數。
- 有效 thread 定義為至少一支 ticker。原始 thread 全部保留於診斷，包括排除後為零者；不直接產生新的清理資料。
- 主分布、門檻比例及 pair 分布以全部原始 threads 為母體，包含零值。另提供僅有效 threads 的 ticker 數分布。門檻比例的分母固定為原始 thread 數。
- 百分位數採 NumPy linear interpolation；同分 ticker 按字母排序，前 20 threads 同分按 thread_id 降冪排序。
- potential_pairs 是各 thread 的 n*(n-1)//2；同一股票 pair 出現在不同 thread 會重複貢獻，不等於全域 unique edges，也不是 rolling network。
- 分批讀取，只在 RAM 保留 ticker 計數、每個 thread 的整數統計及前 20 名資料，不建立 pair 明細。
- 作者不在此分析投影中，unique authors 標為 N/A；日期為 thread 的 first_timestamp，使用 UTC。
- 新 run 只輸出簡潔比較表與一份完整 JSON 報告。既有審核表、03 輸出及 pipeline 指向均不改動。


## ???after ??
- before ?????? threads?after ? ticker ????????? potential pair ????????? ticker ? ?1 ? threads?`population_threads` ???????`tickers_*` ?????????
- after ???? ticker ?? 0????????????????? ticker ????? impact ? `zero_ticker_threads_after`?
- ?????????? ?2 ?? <2?? thread ??????????? threads????????????mentions ? pair ???????? before ??????


In [1]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import json, hashlib, heapq, time, csv, io
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'Data/process').is_dir())
REVIEW_FILE=ROOT/'Data/process/t_04_ticker_review/2026-09-22_09-42-10_UTC/top200_ticker_review.csv'
review_bytes=REVIEW_FILE.read_bytes()
parsed=list(csv.reader(io.StringIO(review_bytes.decode('utf-8-sig')),strict=True))
header=parsed[0]
review_records=[]
trailing_empty_repairs=[]
for line,row in enumerate(parsed[1:],2):
    if len(row)>len(header) and all(value=='' for value in row[len(header):]):
        trailing_empty_repairs.append(line)
        row=row[:len(header)]
    assert len(row)==len(header), ('Unexpected review CSV fields',line)
    review_records.append(row)
review=pd.DataFrame(review_records,columns=header)
review_meta=json.loads((REVIEW_FILE.parent/'run_summary.json').read_text(encoding='utf-8'))
assert review.ticker.is_unique and review.exclude.str.strip().isin(['0','1']).all()
assert set(review.ticker)=={r['ticker'] for r in review_meta['candidates']}, '審核 ticker 名單與原始候選不同'
assert review.ticker.eq(review.ticker.str.strip().str.upper()).all()
excluded_rows=review.loc[review.exclude.str.strip().eq('1')].copy()
EXCLUDED=set(excluded_rows.ticker)
assert EXCLUDED, '沒有 exclude=1 的 ticker，請先人工標記'
SOURCE=Path(review_meta['source'])
source_stat=[SOURCE.stat().st_size,SOURCE.stat().st_mtime_ns]
assert source_stat==review_meta['source_size_mtime'], '03 來源已變更，應重新審核'
OUT=ROOT/'Data/process/t04_1_ticker_exclusion_diagnostics'/datetime.now(timezone.utc).strftime('%Y-%m-%d_%H-%M-%S_UTC')
OUT.mkdir(parents=True,exist_ok=False)
print('來源：',SOURCE)
print('排除 ticker 數：',len(EXCLUDED),'；未填理由數：',int(excluded_rows.reason.str.strip().eq('').sum()))
print('輸出：',OUT)


來源： C:\Users\eagle\RedditStock\Data\process\03_thread_construction\2026-09-22_03-56-59_UTC\threads.parquet
排除 ticker 數： 180 ；未填理由數： 177
輸出： C:\Users\eagle\RedditStock\Data\process\t04_1_ticker_exclusion_diagnostics\2026-09-22_10-06-32_UTC


## 1. 分批比較 thread ticker 集合
每個 thread 先去重，再移除人工指定 ticker。保留全體 before/after 數量與前 20 名完整 ticker 清單。

In [2]:
started=time.perf_counter()
before_frequency=Counter();after_frequency=Counter()
before_counts=[];after_counts=[]
top_before=[];top_after=[]
previous=None
date_min=date_max=None
def retain_top(heap, count, thread, first, tickers):
    item=(count,thread,first,tickers)
    if len(heap)<20: heapq.heappush(heap,item)
    elif item[:2]>heap[0][:2]: heapq.heapreplace(heap,item)
for batch_no,batch in enumerate(pq.ParquetFile(SOURCE).iter_batches(batch_size=5000,columns=['thread_id','first_timestamp','tickers_in_thread']),1):
    dates=[];batch_before=0;batch_after=0
    for row in batch.to_pylist():
        thread=row['thread_id']
        assert thread and (previous is None or previous<thread), '輸入需為03已排序、唯一的thread表'
        previous=thread
        before=sorted(set(row['tickers_in_thread']))
        after=[t for t in before if t not in EXCLUDED]
        before_frequency.update(before);after_frequency.update(after)
        before_counts.append(len(before));after_counts.append(len(after))
        batch_before+=len(before);batch_after+=len(after)
        first=row['first_timestamp'].isoformat() if row['first_timestamp'] is not None else None
        if first: dates.append(first)
        retain_top(top_before,len(before),thread,first,before)
        if after: retain_top(top_after,len(after),thread,first,after)
    if dates:
        date_min=min(date_min,min(dates)) if date_min else min(dates)
        date_max=max(date_max,max(dates)) if date_max else max(dates)
    print({'batch':batch_no,'row_count':batch.num_rows,'unique_threads':batch.num_rows,'unique_authors':'N/A',
           'date_min':min(dates) if dates else None,'date_max':max(dates) if dates else None,
           'mentions_before':batch_before,'mentions_after':batch_after})
before=np.asarray(before_counts,dtype=np.int64);after=np.asarray(after_counts,dtype=np.int64)
assert len(before)==pq.read_metadata(SOURCE).num_rows and len(before)>0
assert np.all(after<=before) and set(after_frequency).isdisjoint(EXCLUDED)
removed=before-after
pairs_before=before*(before-1)//2;pairs_after=after*(after-1)//2
assert int(before.sum())==sum(before_frequency.values())
assert int(after.sum())==sum(after_frequency.values())


{'batch': 1, 'row_count': 5000, 'unique_threads': 5000, 'unique_authors': 'N/A', 'date_min': '2012-09-17T20:19:16+00:00', 'date_max': '2015-08-11T02:19:13+00:00', 'mentions_before': 91371, 'mentions_after': 11812}
{'batch': 2, 'row_count': 5000, 'unique_threads': 5000, 'unique_authors': 'N/A', 'date_min': '2015-06-27T15:32:41+00:00', 'date_max': '2016-02-01T08:45:52+00:00', 'mentions_before': 90707, 'mentions_after': 12473}


{'batch': 3, 'row_count': 5000, 'unique_threads': 5000, 'unique_authors': 'N/A', 'date_min': '2015-11-05T13:50:23+00:00', 'date_max': '2016-07-19T19:29:06+00:00', 'mentions_before': 96517, 'mentions_after': 12833}
{'batch': 4, 'row_count': 5000, 'unique_threads': 5000, 'unique_authors': 'N/A', 'date_min': '2016-02-25T17:49:11+00:00', 'date_max': '2016-05-20T17:20:26+00:00', 'mentions_before': 88912, 'mentions_after': 12417}


{'batch': 5, 'row_count': 5000, 'unique_threads': 5000, 'unique_authors': 'N/A', 'date_min': '2016-05-09T16:51:37+00:00', 'date_max': '2017-02-11T23:37:18+00:00', 'mentions_before': 86029, 'mentions_after': 12296}
{'batch': 6, 'row_count': 5000, 'unique_threads': 5000, 'unique_authors': 'N/A', 'date_min': '2016-07-14T03:01:21+00:00', 'date_max': '2017-01-25T04:38:10+00:00', 'mentions_before': 80420, 'mentions_after': 12236}


{'batch': 7, 'row_count': 5000, 'unique_threads': 5000, 'unique_authors': 'N/A', 'date_min': '2016-08-31T14:03:00+00:00', 'date_max': '2017-02-27T15:24:47+00:00', 'mentions_before': 77181, 'mentions_after': 10301}
{'batch': 8, 'row_count': 5000, 'unique_threads': 5000, 'unique_authors': 'N/A', 'date_min': '2016-10-25T16:17:32+00:00', 'date_max': '2016-12-17T03:09:28+00:00', 'mentions_before': 71241, 'mentions_after': 10025}


{'batch': 9, 'row_count': 5000, 'unique_threads': 5000, 'unique_authors': 'N/A', 'date_min': '2016-12-14T18:11:21+00:00', 'date_max': '2017-03-27T17:47:00+00:00', 'mentions_before': 82162, 'mentions_after': 11652}
{'batch': 10, 'row_count': 5000, 'unique_threads': 5000, 'unique_authors': 'N/A', 'date_min': '2017-02-09T22:06:37+00:00', 'date_max': '2017-04-15T17:48:36+00:00', 'mentions_before': 96144, 'mentions_after': 13535}


...

{'batch': 47, 'row_count': 5000, 'unique_threads': 5000, 'unique_authors': 'N/A', 'date_min': '2020-05-18T15:47:23+00:00', 'date_max': '2020-05-30T22:41:35+00:00', 'mentions_before': 89658, 'mentions_after': 19026}


{'batch': 48, 'row_count': 5000, 'unique_threads': 5000, 'unique_authors': 'N/A', 'date_min': '2020-05-28T17:38:54+00:00', 'date_max': '2020-08-01T01:41:29+00:00', 'mentions_before': 99944, 'mentions_after': 24773}


{'batch': 49, 'row_count': 5000, 'unique_threads': 5000, 'unique_authors': 'N/A', 'date_min': '2020-06-11T23:00:44+00:00', 'date_max': '2020-07-12T18:52:02+00:00', 'mentions_before': 103997, 'mentions_after': 22725}


{'batch': 50, 'row_count': 5000, 'unique_threads': 5000, 'unique_authors': 'N/A', 'date_min': '2020-06-25T04:43:49+00:00', 'date_max': '2020-07-12T21:17:37+00:00', 'mentions_before': 114143, 'mentions_after': 27014}


{'batch': 51, 'row_count': 5000, 'unique_threads': 5000, 'unique_authors': 'N/A', 'date_min': '2020-07-12T05:37:57+00:00', 'date_max': '2020-07-31T02:35:33+00:00', 'mentions_before': 115406, 'mentions_after': 28077}
{'batch': 52, 'row_count': 3609, 'unique_threads': 3609, 'unique_authors': 'N/A', 'date_min': '2012-04-11T16:46:43+00:00', 'date_max': '2020-08-06T01:21:55+00:00', 'mentions_before': 76314, 'mentions_after': 16737}


## 2. Before / after 比較與污染影響
分布包含零 ticker thread；另列有效 threads 的分布。所有 ratio 為 0～1，delta 為 after 減 before。

In [3]:
def distribution(values):
    if not len(values): return dict.fromkeys(['min','max','mean','median','P75','P90','P95','P99'])
    return {'min':int(values.min()),'max':int(values.max()),'mean':float(values.mean()),
            'median':float(np.median(values)),**{f'P{p}':float(np.percentile(values,p,method='linear')) for p in [75,90,95,99]}}
def thread_metrics(values):
    result={'population_threads':len(values),'valid_threads':int((values>0).sum()),
            **{'tickers_'+k:v for k,v in distribution(values).items()},
            **{'valid_threads_tickers_'+k:v for k,v in distribution(values[values>0]).items()}}
    for name,mask in [('zero',values==0),('one',values==1),('ge2',values>=2),('ge5',values>=5),('ge10',values>=10),('ge20',values>=20)]:
        result[name+'_threads']=int(mask.sum());result[name+'_ratio']=float(mask.mean()) if len(values) else None
    return result
left={'unique_tickers':len(before_frequency),'ticker_mentions':int(before.sum()),**thread_metrics(before),
      'potential_pairs_total':int(pairs_before.sum()),**{'potential_pairs_'+k:v for k,v in distribution(pairs_before).items()}}
after_valid=after[after>=1]
pairs_after_valid=pairs_after[after>=1]
right={'unique_tickers':len(after_frequency),'ticker_mentions':int(after.sum()),**thread_metrics(after_valid),
       'potential_pairs_total':int(pairs_after.sum()),**{'potential_pairs_'+k:v for k,v in distribution(pairs_after_valid).items()}}
summary=pd.DataFrame([{'metric':k,'before':left[k],'after':right[k],
                       'delta':right[k]-left[k] if left[k] is not None and right[k] is not None else None} for k in left])
removed_mentions=sum(before_frequency[t] for t in EXCLUDED)
assert removed_mentions==int(removed.sum())
impact={'flagged_tickers':len(EXCLUDED),'excluded_tickers_present':len(EXCLUDED & set(before_frequency)),
        'excluded_tickers_absent':sorted(EXCLUDED-set(before_frequency)),
        'removed_mentions':removed_mentions,
        'removed_mention_ratio':removed_mentions/int(before.sum()) if before.sum() else None,
        'mentions_change_ratio':(int(after.sum())-int(before.sum()))/int(before.sum()) if before.sum() else None,
        'threads_became_zero':int(((before>0)&(after==0)).sum()),
        'threads_ge2_became_lt2':int(((before>=2)&(after<2)).sum()),
        'removed_tickers_per_thread':distribution(removed),
        'impact_population_threads':len(before),
        'zero_ticker_threads_before':int((before==0).sum()),
        'zero_ticker_threads_after':int((after==0).sum()),
        'potential_pairs_reduction_ratio':1-int(pairs_after.sum())/int(pairs_before.sum()) if pairs_before.sum() else None}
with pd.option_context('display.max_rows',None): display(summary)
display(pd.DataFrame([impact]))

assert right["population_threads"]==int((after>=1).sum())
assert right["one_threads"]+right["ge2_threads"]==right["population_threads"]
assert int(pairs_after_valid.sum())==int(pairs_after.sum())


,metric,before,after,delta
0,unique_tickers,5.682000e+03,5.502000e+03,-1.800000e+02
1,ticker_mentions,5.146535e+06,9.532020e+05,-4.193333e+06
2,population_threads,2.586090e+05,1.600580e+05,-9.855100e+04
3,valid_threads,2.586090e+05,1.600580e+05,-9.855100e+04
4,tickers_min,1.000000e+00,1.000000e+00,0.000000e+00
5,tickers_max,1.052000e+03,8.720000e+02,-1.800000e+02
6,tickers_mean,1.990083e+01,5.955354e+00,-1.394548e+01
7,tickers_median,1.100000e+01,2.000000e+00,-9.000000e+00
8,tickers_P75,2.400000e+01,5.000000e+00,-1.900000e+01
9,tickers_P90,4.300000e+01,1.000000e+01,-3.300000e+01


,flagged_tickers,excluded_tickers_present,excluded_tickers_absent,removed_mentions,removed_mention_ratio,mentions_change_ratio,threads_became_zero,threads_ge2_became_lt2,removed_tickers_per_thread,impact_population_threads,zero_ticker_threads_before,zero_ticker_threads_after,potential_pairs_reduction_ratio
0,180,180,[],4193333,0.814788,-0.814788,98551,135950,"{'min': 0, 'max': 180, 'mean': 16.214953849247...",258609,0,98551,0.756178


## 3. 排除清單、前 50 名 ticker 與前 20 個 threads
兩個版本各自重新排名；thread 清單中附完整 ticker list，方便觀察大型 thread 是否仍集中大量 ticker。

In [4]:
def top_tickers(counter):
    return [{'rank':i,'ticker':t,'mention_frequency':count} for i,(t,count) in enumerate(sorted(counter.items(),key=lambda item:(-item[1],item[0]))[:50],1)]
def top_threads(heap):
    return [{'rank':i,'thread_id':t,'first_timestamp_utc':first,'unique_tickers':count,
             'potential_pairs':count*(count-1)//2,'tickers':tickers}
            for i,(count,t,first,tickers) in enumerate(sorted(heap,reverse=True),1)]
excluded_report=[{'ticker':row.ticker,'mention_frequency_before':before_frequency[row.ticker],'reason':row.reason}
                 for row in excluded_rows.sort_values('ticker').itertuples(index=False)]
rankings={'top50_tickers_before':top_tickers(before_frequency),'top50_tickers_after':top_tickers(after_frequency),
          'top20_threads_before':top_threads(top_before),'top20_threads_after':top_threads(top_after)}
print('被排除 ticker：')
with pd.option_context('display.max_rows',None): display(pd.DataFrame(excluded_report))
for label,records in rankings.items():
    print(label)
    with pd.option_context('display.max_rows',50,'display.max_colwidth',None): display(pd.DataFrame(records))
assert SOURCE.stat().st_size==source_stat[0] and SOURCE.stat().st_mtime_ns==source_stat[1]
assert REVIEW_FILE.read_bytes()==review_bytes
summary.to_csv(OUT/'before_after_summary.csv',index=False,encoding='utf-8-sig')
report={'status':'complete','source':str(SOURCE),'source_size_mtime':source_stat,'review_file':str(REVIEW_FILE),
        'review_sha256':hashlib.sha256(review_bytes).hexdigest(),'review_trailing_empty_fields_ignored_at_records':trailing_empty_repairs,
        'mention_definition':'one unique ticker per thread','ratio_denominator':{'before':'all original threads','after':'threads with at least one ticker after exclusion','impact':'all original threads; reduction ratios use before totals'},
        'date_min':date_min,'date_max':date_max,'unique_authors':None,
        'before':left,'after':right,'impact':impact,'excluded_tickers':excluded_report,**rankings,
        'processing_seconds':time.perf_counter()-started,'pairs_materialized':False}
(OUT/'diagnostic_report.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
print('完成：',OUT)


被排除 ticker：


,ticker,mention_frequency_before,reason
0,ACT,6718,
1,AGE,7267,
2,AGO,35426,
3,AIR,5295,
4,ALL,120858,
5,ALOT,4361,
6,ANY,87992,
7,APP,12496,
8,ARE,142391,
9,ATH,6165,


top50_tickers_before


,rank,ticker,mention_frequency
0,1,YOU,202685
1,2,FOR,176048
2,3,ARE,142391
3,4,ALL,120858
4,5,OUT,114703
5,6,MORE,110962
6,7,NOW,101664
7,8,HAS,100678
8,9,GOOD,95276
9,10,ONE,93341


top50_tickers_after


...

top20_threads_before


,rank,thread_id,first_timestamp_utc,unique_tickers,potential_pairs,tickers
0,1,gxbzbj,2020-06-05T20:07:29+00:00,1052,552826,"[AAL, AAPL, AAXN, ABB, ABBV, ABC, ABT, ACB, ACRE, ACT, ADBE, ADS, ADSK, ADT, AEG, AER, AERI, AGE, AGNC, AGO, AHH, AIG, AIM, AIN, AIR, ALE, ALEX, ALGN, ALGT, ALK, ALL, ALLY, ALOT, ALT, ALTA, AMAT, AMC, AMD, AMG, AMGN, AMID, AMN, AMP, AMZN, ANY, AOBC, APA, APEX, APHA, APP, APPN, APPS, APR, APT, APY, AQST, AQUA, ARCH, ARE, ASH, ATEC, ATH, ATVI, ATX, AUD, AUG, AURA, AUS, AUTO, AUY, AVG, AWH, AXE, AXP, AZZ, BABA, BABY, BAC, BAH, BAM, BAND, BARE, BARK, BAS, BASE, BBBY, BBQ, BBVA, BBW, BBY, BDAY, BEAM, BEAR, BEAT, BEER, BEN, BENE, BFB, BFS, BGG, ...]"
1,2,hgeyf9,2020-06-26T20:07:36+00:00,951,451725,"[AAL, AAPL, ABBV, ABC, ABT, ACB, ACC, ACH, ACI, ACT, ADBE, ADM, ADS, ADV, AGE, AGO, AGS, AHH, AHPI, AHS, AIG, AIM, AIN, AIR, AKAM, AKS, ALE, ALEX, ALK, ALL, ALLY, ALOT, ALT, ALTO, AMC, AMD, AMG, AMID, AMP, AMT, AMZN, ANDS, ANET, ANY, APDN, APEX, APHA, APP, APPS, APR, APT, ARC, ARDS, ARE, ASRT, ATH, ATHX, ATVI, AUD, AUG, AUS, AUTO, AUY, AVG, AVID, AWH, AXE, AXO, AXP, BABA, BABY, BAC, BAH, BAM, BANC, BAND, BARE, BAS, BASE, BBBY, BBCP, BBG, BBQ, BBW, BBY, BDAY, BEAM, BEAR, BEAT, BEER, BEN, BFS, BID, BIG, BILL, BIO, BIRD, BIT, BITE, BLK, ...]"
2,3,h7s30c,2020-06-12T20:07:30+00:00,930,431985,"[AAL, AAPL, AAXN, ABBV, ABC, ABT, ACB, ACC, ACH, ACRE, ACT, ACTT, ADBE, ADS, AFG, AFIB, AGE, AGO, AHH, AIG, AIKI, AIM, AIN, AIR, ALEX, ALK, ALL, ALLY, ALOT, ALT, AMC, AMD, AMG, AMID, AMP, AMZN, ANDS, ANY, APC, APEX, APHA, APM, APP, APPS, APT, AQUA, ARC, ARCH, ARE, ARKR, ARMK, ARW, ASH, ASPN, ATA, ATH, ATVI, AUD, AUG, AURA, AUS, AUTO, AVCT, AVG, AVID, AXE, AXON, AXP, AZEK, BABA, BABY, BAC, BAH, BAM, BAND, BAP, BARE, BARK, BASE, BBG, BBL, BBQ, BBW, BBY, BDAY, BDC, BEAM, BEAR, BEAT, BEER, BEN, BFS, BHP, BID, BIG, BILL, BIO, BIOX, BIRD, BIT, ...]"
3,4,gt08v2,2020-05-29T20:07:15+00:00,921,423660,"[AAL, AAPL, AAXN, ABBV, ABC, ABT, ACAT, ACB, ACC, ACER, ACH, ACT, ADBE, ADI, ADS, ADSK, AEO, AEP, AES, AGE, AGEN, AGO, AHH, AIG, AIM, AIN, AIR, AJRD, ALE, ALEX, ALK, ALL, ALLY, ALOT, ALT, AMC, AMD, AMG, AMID, AMP, AMZN, ANY, AOBC, APAC, APEX, APHA, APO, APOG, APP, APPS, APT, ARC, ARCH, ARE, AREX, ARI, ARLO, ARWR, ARYA, ASH, ATH, ATVI, AUG, AURA, AUTO, AUY, AVEO, AVG, AVGO, AXON, AXP, AYTU, AZZ, BABA, BABY, BAC, BAH, BAM, BAND, BARE, BARK, BAS, BASE, BBQ, BBY, BDAY, BEAM, BEAR, BEAT, BEEM, BEER, BEN, BFB, BFS, BGG, BID, BIG, BILL, BIO, BIRD, ...]"
4,5,how5ir,2020-07-10T20:07:32+00:00,907,410871,"[AAL, AAPL, ABBV, ABT, ACAT, ACC, ACH, ACLS, ACRE, ACT, ADS, ADT, AGE, AGO, AHH, AIM, AIN, AIR, ALE, ALEX, ALL, ALLY, ALOT, ALT, AMC, AMCS, AMD, AMG, AMID, AMP, AMT, AMZN, ANGO, ANY, APA, APEX, APHA, APP, APPS, APPX, APR, APRN, APT, APY, ARA, ARC, ARCE, ARCH, ARDX, ARE, ARMK, ARYA, ASA, ATH, ATVI, AUG, AUS, AUTO, AUY, AVG, AVID, AVXL, AWH, AXE, AXP, AXU, AZZ, BABA, BABY, BAC, BAH, BAM, BAND, BARE, BARK, BASE, BBBY, BBQ, BBW, BBY, BCRX, BDAY, BDN, BEAM, BEAR, BEAT, BEER, BEN, BFS, BGG, BID, BIER, BIG, BILL, BIO, BIOS, BIRD, BIT, BITE, BKK, ...]"
5,6,hc7rvq,2020-06-19T20:07:30+00:00,906,409965,"[AAL, AAPL, ABC, ABT, ACB, ACC, ACER, ACRE, ACT, ACTT, ADBE, ADNT, ADS, AFL, AGE, AGO, AGS, AHH, AIM, AIN, AIR, ALE, ALEX, ALF, ALL, ALLT, ALLY, ALOT, ALT, AMAT, AMC, AMD, AMED, AMID, AMN, AMP, AMS, AMZN, ANF, ANY, APEX, APHA, APP, APPS, APT, APTV, ARC, ARCH, ARE, ARMK, ASA, ASH, ATA, ATH, ATR, ATVI, AUG, AUTO, AVA, AVG, AWK, AXE, AXP, AYX, AZZ, BABA, BABY, BAC, BAH, BAM, BAND, BARE, BARK, BAS, BASE, BBD, BBG, BBQ, BBW, BBY, BDAY, BEAM, BEAR, BEAT, BEER, BEN, BID, BIG, BILI, BILL, BIO, BIOS, BIP, BIRD, BIT, BITE, BLOG, BLVD, BMO, BMY, ...]"
6,7,ht2v2k,2020-07-17T20:07:40+00:00,897,401856,"[AAL, AAME, AAP, AAPL, ABBV, ABT, ACB, ACC, ACHV, ACI, ACRE, ACT, ADBE, ADS, ADT, AEO, AFMD, AGE, AGO, AGRX, AHH, AIG, AIM, AIMT, AIN, AIR, AIRS, AKAM, ALC, ALE, ALEX, ALL, ALLY, ALOT, ALT, AMC, AMD, AMID, AMP, AMT, AMZN, ANDA, ANGI, ANY, APEX, APHA, APP, APPS, APR, APRN, APT, APY, 

top20_threads_after


,rank,thread_id,first_timestamp_utc,unique_tickers,potential_pairs,tickers
0,1,gxbzbj,2020-06-05T20:07:29+00:00,872,379756,"[AAL, AAPL, AAXN, ABB, ABBV, ABC, ABT, ACB, ACRE, ADBE, ADS, ADSK, ADT, AEG, AER, AERI, AGNC, AHH, AIG, AIM, AIN, ALE, ALEX, ALGN, ALGT, ALK, ALLY, ALT, ALTA, AMAT, AMC, AMD, AMG, AMGN, AMID, AMN, AMP, AMZN, AOBC, APA, APEX, APHA, APPN, APPS, APR, APT, APY, AQST, AQUA, ARCH, ASH, ATEC, ATVI, ATX, AUD, AUG, AURA, AUS, AUY, AWH, AXE, AXP, AZZ, BABA, BAC, BAH, BAM, BAND, BARE, BARK, BAS, BBBY, BBQ, BBVA, BBW, BBY, BDAY, BEAM, BEN, BENE, BFB, BFS, BGG, BIO, BIRD, BITE, BKNG, BKR, BLK, BLMN, BLOG, BLU, BMO, BOI, BOLD, BOLT, BOOT, BORR, BOWL, BRAG, ...]"
1,2,hgeyf9,2020-06-26T20:07:36+00:00,771,296835,"[AAL, AAPL, ABBV, ABC, ABT, ACB, ACC, ACH, ACI, ADBE, ADM, ADS, ADV, AGS, AHH, AHPI, AHS, AIG, AIM, AIN, AKAM, AKS, ALE, ALEX, ALK, ALLY, ALT, ALTO, AMC, AMD, AMG, AMID, AMP, AMT, AMZN, ANDS, ANET, APDN, APEX, APHA, APPS, APR, APT, ARC, ARDS, ASRT, ATHX, ATVI, AUD, AUG, AUS, AUY, AVID, AWH, AXE, AXO, AXP, BABA, BAC, BAH, BAM, BANC, BAND, BARE, BAS, BBBY, BBCP, BBG, BBQ, BBW, BBY, BDAY, BEAM, BEN, BFS, BIO, BIRD, BITE, BLK, BLMN, BLNK, BLOG, BMRG, BMY, BOI, BOLD, BOLT, BOOT, BOWL, BRAG, BREW, BRKA, BRKS, BROS, BSX, BULK, BURL, BYND, CAG, CAKE, ...]"
2,3,h7s30c,2020-06-12T20:07:30+00:00,750,280875,"[AAL, AAPL, AAXN, ABBV, ABC, ABT, ACB, ACC, ACH, ACRE, ACTT, ADBE, ADS, AFG, AFIB, AHH, AIG, AIKI, AIM, AIN, ALEX, ALK, ALLY, ALT, AMC, AMD, AMG, AMID, AMP, AMZN, ANDS, APC, APEX, APHA, APM, APPS, APT, AQUA, ARC, ARCH, ARKR, ARMK, ARW, ASH, ASPN, ATA, ATVI, AUD, AUG, AURA, AUS, AVCT, AVID, AXE, AXON, AXP, AZEK, BABA, BAC, BAH, BAM, BAND, BAP, BARE, BARK, BBG, BBL, BBQ, BBW, BBY, BDAY, BDC, BEAM, BEN, BFS, BHP, BIO, BIOX, BIRD, BITE, BLDP, BLMN, BLOG, BMI, BMO, BOAS, BOI, BOLD, BON, BOOT, BOWL, BRAG, BREW, BRKA, BROS, BSN, BULK, BYND, CAKE, CAL, ...]"
3,4,gt08v2,2020-05-29T20:07:15+00:00,741,274170,"[AAL, AAPL, AAXN, ABBV, ABC, ABT, ACAT, ACB, ACC, ACER, ACH, ADBE, ADI, ADS, ADSK, AEO, AEP, AES, AGEN, AHH, AIG, AIM, AIN, AJRD, ALE, ALEX, ALK, ALLY, ALT, AMC, AMD, AMG, AMID, AMP, AMZN, AOBC, APAC, APEX, APHA, APO, APOG, APPS, APT, ARC, ARCH, AREX, ARI, ARLO, ARWR, ARYA, ASH, ATVI, AUG, AURA, AUY, AVEO, AVGO, AXON, AXP, AYTU, AZZ, BABA, BAC, BAH, BAM, BAND, BARE, BARK, BAS, BBQ, BBY, BDAY, BEAM, BEEM, BEN, BFB, BFS, BGG, BIO, BIRD, BITE, BLE, BLK, BLOG, BLU, BMCH, BMI, BOI, BOLD, BOLT, BOOT, BOWL, BRAG, BREW, BRKA, BROS, BTI, BULK, BWC, BYD, ...]"
4,5,how5ir,2020-07-10T20:07:32+00:00,727,263901,"[AAL, AAPL, ABBV, ABT, ACAT, ACC, ACH, ACLS, ACRE, ADS, ADT, AHH, AIM, AIN, ALE, ALEX, ALLY, ALT, AMC, AMCS, AMD, AMG, AMID, AMP, AMT, AMZN, ANGO, APA, APEX, APHA, APPS, APPX, APR, APRN, APT, APY, ARA, ARC, ARCE, ARCH, ARDX, ARMK, ARYA, ASA, ATVI, AUG, AUS, AUY, AVID, AVXL, AWH, AXE, AXP, AXU, AZZ, BABA, BAC, BAH, BAM, BAND, BARE, BARK, BBBY, BBQ, BBW, BBY, BCRX, BDAY, BDN, BEAM, BEN, BFS, BGG, BIER, BIO, BIOS, BIRD, BITE, BKK, BKR, BLMN, BLNK, BLOG, BLVD, BMO, BMY, BNGO, BOI, BOLD, BOLT, BOOT, BOWL, BOXL, BRAG, BREW, BRKA, BROS, BULK, BURL, BYND, ...]"
5,6,hc7rvq,2020-06-19T20:07:30+00:00,726,263175,"[AAL, AAPL, ABC, ABT, ACB, ACC, ACER, ACRE, ACTT, ADBE, ADNT, ADS, AFL, AGS, AHH, AIM, AIN, ALE, ALEX, ALF, ALLT, ALLY, ALT, AMAT, AMC, AMD, AMED, AMID, AMN, AMP, AMS, AMZN, ANF, APEX, APHA, APPS, APT, APTV, ARC, ARCH, ARMK, ASA, ASH, ATA, ATR, ATVI, AUG, AVA, AWK, AXE, AXP, AYX, AZZ, BABA, BAC, BAH, BAM, BAND, BARE, BARK, BAS, BBD, BBG, BBQ, BBW, BBY, BDAY, BEAM, BEN, BILI, BIO, BIOS, BIP, BIRD, BITE, BLOG, BLVD, BMO, BMY, BOI, BOLD, BOLT, BON, BOOT, BOWL, BRAG, BREW, BRKA, BROS, BSX, BTU, BULK, BYFC, BYND, CAB, CAH, CAKE, CAL, CALC, CALI, ...]"
6,7,ht2v2k,2020-07-17T20:07:40+00:00,717,256686,"[AAL, AAME, AAP, AAPL, ABBV, ABT, ACB, ACC, ACHV, ACI, ACRE, ADBE, ADS, ADT, AEO, AFMD, AGRX, AHH, AIG, AIM, AIMT, AIN, AIRS, AKAM, ALC, ALE, ALEX, ALLY, ALT, AMC, AMD, AMID, AMP, AMT, AMZN, ANDA, ANGI, APEX, APHA, APPS, APR, APRN, APT, APY, ARC

完成： C:\Users\eagle\RedditStock\Data\process\t04_1_ticker_exclusion_diagnostics\2026-09-22_10-06-32_UTC
